In [1]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

In [2]:
# Load the model on the available device(s)
with torch.no_grad():
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2.5-VL-72B-Instruct-AWQ",
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map="auto",
    )

    # Load the processor
    processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-72B-Instruct-AWQ")

print("Model and Processor Loaded Successfully!")

We suggest you to set `torch_dtype=torch.float16` for better efficiency with AWQ.


Loading checkpoint shards:   0%|          | 0/11 [00:00<?, ?it/s]

Model and Processor Loaded Successfully!


In [3]:
# im_name = "../data/inputs/IMG_1029.jpg"

# messages = [
#     {
#         "role": "user",
#         "content": [
#             {"type": "image", "image": im_name},
#             {
#                 "type": "text",
#                 "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français. Votre mission est d'identifier diverses informations décrites plus loin. Identifiez uniquement le texte manuscrit.

# La première information consiste en l'identification du nom complet de la personne ainsi que le nom du mari. La première ligne suit ce format : "Nom Prénom1 (Prénom2) (Prénom3) [Marqueur] Nom_mari (Prénom_mari)"
# Voici des exemples de marqueurs : 'fme', 'fe', 'vve', 've', 'femme', 'divorcée', etc

# La sortie XML attendue est structurée comme suit :

#     <nom_complet>Nom Prénoms</nom_complet>
#     <nom_complet_mari>Nom</nom_complet_mari>
#     <marqueur>marqueur</marqueur>

# Cas spécifiques :

#     La sortie ne doit pas contenir de date ou autre informations.
#     Le nom complet de l'individu se trouve obligatoirement avant le marqueur.
#     Le nom complet mari se trouve obligatoirement après le marqueur.
#     Ecrire uniquement les informations relative aux différents champs, sans aucun autre texte.
#     N'écrire que le XML, sans aucun autre texte.

# La seconde information consiste en l'identification de la date de naissance de la personne.
# La sortie XML attendue est structurée comme suit :
    
#     <date_naissance>date</date_naissance>
#     <lieu_naissance>lieu_naissance</lieu_naissance>

# Cas spécifiques :
#     La date de naissance peut être sous la forme "jj/mm/aaaa" ou "jj mois aaaa".
#     Le lieu de naissance peut être une ville, un village, un pays, etc.
#     Le lieu de naissance peut être une ville, un village, un pays, etc se situant dans un autre pays.

# La troisième information consiste en l'identification de la nationalité de la personne.
# La sortie XML attendue est structurée comme suit :

#     <nationalite>nationalite</nationalite>

# Cas spécifiques :
#     Il est possible que la personne ait plusieurs nationalités. Dans ce cas indiquez également la date de naturalisation si elle est présente.
#     N'écrire que le XML, sans aucun autre texte.

# La quatrieme information consiste en l'identification de l'etat civil de la personne.
# La sortie XML attendue est structurée comme suit :

#     <status>status</status>
#     <adresse>adresse</adresse>
#     <ville>ville</ville>

# Cas spécifiques :
#     Le status peut être au masculin ou au féminin.
#     La ville peut être une ville, un village, un pays, etc.
#     La ville se situe obligatoirement en France.
#     Le nom de la ville est optionnel.
#     Si la ligne "Etat civil" est présente, l'adresse s'y trouve obligatoirement.
#     L'adresse doit contenir le nom de la rue et un numéro.
#     N'écrire que le JSON, sans aucun autre texte.

# La cinquième information consiste en l'identification des éléments de la table et des annotations/renseignements.
# La sortie XML attendue pour la table est structurée comme suit :

#     <table>
#         <ligne>
#             <atelier (optionnel)>element1</atelier>
#             <occupation>element2</occupation>
#             <entree>element3</entree>
#             <sortie>element4</sortie>
#             <presence><annees>X</annees><mois>Y</mois><jours (optionnel)>Z</jours></presence>
#             <observations>element5</observations>
#             <divers (optionnel)>element6</divers>
#             ...
#         </ligne>
#     </table>

# Cas spécifiques :
#     Si une cellule d'une ligne est vide ou inexistante, ecrire "".
#     Nommer les balises elements en fonction du contenu de l'entete de la colonne.
#     N'écrire que le XML, sans aucun autre texte.

# La sortie XML attendue pour les annotation est structurée comme suit :

#     <annotation>annotation</annotation>

# Cas spécifiques :
#     Il est possible que l'annotation ou le renseignement soit absent.
#     Les annotations peuvent être positionnées à différents endroits du document.
#     Les annotations peuvent tenir sur plusieurs lignes.
#     N'écrire que le XML, sans aucun autre texte.

# Pour finir il faut veiller à bien fermer les balises XML.
# """
#             },
#         ],
#     }
# ]

In [11]:
# 0-shot

im_name = "../data/inputs/A-B/IMG_0023.jpg"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": im_name},
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous lisez les informations de l'entête du document en reconnaissant les éléments sous la forme ("first level", "second level") :
```plaintext
[("Nom complet", ""),
("Nom complet du Mari", ""),
("Marqueur", ""),
("Date de naissance", ""),
("Lieu de naissance", ""),
("Nationalité", ""),
("Statut marital", ""),
("Genre", ""),
("Adresse", ""),
("Ville", "")]
```
Contexte :
- Le document est une fiche d'entreprise manuscrite en français datant des années 1900.
- Les noms des personnes et des villes ne font pas toujours français.
- Les marqueurs peuvent être 'fme', 'fe', 'vve', 've', 'femme', 'divorcée', etc.
- Les marqueurs précèdent le nom du mari si la personne est une femme mariée.
- La date de naissance peut être sous la forme "jj/mm/aaaa" ou "jj mois aaaa".
- Le lieu de naissance peut être une ville, un village, un pays, etc.
- Un individu peut avoir plusieurs nationalités (toutes les indiquer).
- Les statuts maritaux doit être 'célibataire', 'marié', 'divorcé', 'veuf', etc ou "" si indéfini.
- Le nom de la ville est optionnel.
- Si la ligne "Etat civil" est présente, l'adresse s'y trouve obligatoirement.
- L'adresse doit contenir le nom de la rue et un numéro.	
- Le genre peut être 'homme' ou 'femme' et est déterminé par la présence d'un e à 'née'.
Tâche :
Reconstruisez, s'il vous plait, l'entête du document en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous lisez l'entête à deux niveau du tableau dans le document en reconnaissant les éléments sous la forme ("first level", "second level") :
```plaintext
[("Atelier", ""),
("Occupation", ""),
("Entrée", ""),
("Sortie", ""),
("Présence: Années", ""),
("Présence: Mois", ""),
("Présence: Jours", ""),
("Observations", ""),
("Divers", "")]
```
Contexte :
- Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
- La colonne 'Atelier' peut être absente.
- Pour chaque ligne du tableau, créer une entrée dans le dictionnaire.
- Il est possible que certaines lignes du tableau ne contiennent qu'une observation. Dans ce cas, remplir uniquement la colonne 'Observations' d'une nouvelle entrée.
- Des lignes peuvent avoir des informations manquantes.
- Tu dois ecrire <bis> à la place de '"'.
- "Divers" contient des informations supplémentaires sur le montant de la gratifiaction par exemple (Francs et centimes).
Tâche :
Reconstruisez, s'il vous plait, le tableau en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. Ne confond pas '"' et '11'. Tu dois egalement faire attention à ne pas confondre une ligne est une annotation marginale.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous prenez connaissance des informations du document en essayant de reperer les informations marginales sous la forme ("first level", "second level") :
```plaintext
[("Annotation", "")]
```
Contexte :
- Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
- Les annotations sont des informations supplémentaires sur le document.
- Les annotations peuvent être des notes, des remarques, des précisions, etc.
- Les annotations sont essentiellement manuscrites.
- Les annotations peuvent être sur plusieurs lignes.
Tâche :
Reconstruisez, s'il vous plait, la liste des annotations en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. Veillez également à ne pas mélanger les annotations.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes un agent spécialisé dans l'adaptation des informations précédemment extraites.
Votre mission est de transformer les informations extraites en XML. Pour cela vous avez a votre disposition les tags suivants :
```xml
<Document>
<Nom>
<Genre>
<Statut>
<DateDeNaissance>
<LieuDeNaissance>
<Nationalité>
<Adresse>
<Ville>
<Table>
<Ligne>
<Atelier>
<Occupation>
<Entrée>
<Sortie>
<Présence>
<Années>
<Mois>
<Jours>
<Observations>
<Divers>
<Annotation>
```
"""
            },
        ],
    }
]

In [19]:
# Few shot example

# 1-shot
shot1_im_name = "IMG_1198"

im_name = "../data/inputs/A-B/IMG_0023.jpg"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": f"../data/inputs/A-B/{shot1_im_name}.jpg"},
            {
                "type": "text",
                "text": """Voici un exemple de la tâche que vous devez accomplir. Vous devez extraire les informations du document manuscrit et les structurer en XML."""
            }
        ]
    },
    {
      "role": "assistant",
      "content": [
          {
                "type": "text", "text": """Voici la transcription correspondant à l'exemple ci-dessus (XML) :
                    <?xml version="1.0" encoding="utf-8"?>
<Document>
  <Nom>Bouteiller Hélène</Nom>
  <Genre>Née</Genre>
  <NomDuConjoint>
    <added-above>
      <sprited>Vve</sprited>
    </added-above>
     
    <sprited>fme Rayot</sprited>
     Fme Braun
  </NomDuConjoint>
  <DateDeNaissance>6 Octobre 1899</DateDeNaissance>
  <LieuDeNaissance>Hérimoncourt</LieuDeNaissance>
  <Nationalité>française</Nationalité>
  <Ville>
    <pence>Belfort</pence>
  </Ville>
  <Adresse>
    <pence>du Berger, 19</pence>
  </Adresse>
  <Table>
    <Ligne>
      <Occupation>
        Fge 
        <pence>cartonnage</pence>
      </Occupation>
      <Entrée>31 Mars 1924</Entrée>
      <Sortie>5 Juillet 1924</Sortie>
      <Présence>3</Présence>
      <Observations>
        5j 
        <printed>
          <coloured-ink>Gratification 1925</coloured-ink>
        </printed>
      </Observations>
      <Divers>89.-</Divers>
    </Ligne>
    <Ligne>
      <Occupation>- d° -</Occupation>
      <Entrée>21 Juillet 1924</Entrée>
      <Sortie>10 Janvier 1930</Sortie>
      <Présence>55</Présence>
      <Observations>
        21 jours 
        <bis/>
        180e ann.
      </Observations>
      <Divers>100.-</Divers>
    </Ligne>
    <Ligne>
      <Observations>
        <bis/>
        1926
      </Observations>
      <Divers>117.-</Divers>
    </Ligne>
    <Ligne>
      <Observations>
        <bis/>
        1927
      </Observations>
      <Divers>126-</Divers>
    </Ligne>
    <Ligne>
      <Observations>
        <bis/>
        10° an. Arm
      </Observations>
      <Divers>140-</Divers>
    </Ligne>
    <Ligne>
      <Observations>
        <bis/>
        1928
      </Observations>
      <Divers>159-</Divers>
    </Ligne>
    <Ligne>
      <Observations>
        <bis/>
        1929
      </Observations>
      <Divers>182-</Divers>
    </Ligne>
    <Ligne>
      <Observations>quitte Belfort</Observations>
    </Ligne>
  </Table>
  <Annotation>
    <pence>Réfectoire 24</pence>
  </Annotation>
</Document>
"""
            },
      ]
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": im_name},
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous lisez les informations de l'entête du document en reconnaissant les éléments sous la forme ("first level", "second level") :
```plaintext
[("Nom complet", ""),
("Nom complet du Mari", ""),
("Marqueur", ""),
("Date de naissance", ""),
("Lieu de naissance", ""),
("Nationalité", ""),
("Statut marital", ""),
("Genre", ""),
("Adresse", ""),
("Ville", "")]
```
Contexte :
- Le document est une fiche d'entreprise manuscrite en français datant des années 1900.
- Le nom de la personne est composé d'un ou plusieurs prénoms et d'un nom de famille.
- Les noms des personnes et des villes ne sont pas toujours français.
- Les marqueurs peuvent être 'fme', 'fe', 'vve', 've', 'femme', 'divorcée', etc.
- Les marqueurs précèdent le nom du mari si la personne est une femme mariée.
- Le nom du conjoint est composé d'un ou plusieurs marqueurs et du ou des noms. ex : "fme Jean Dupont", "ve Marie Curie".
- La date de naissance peut être sous la forme "jj/mm/aaaa" ou "jj mois aaaa".
- Le lieu de naissance peut être une ville, un village, un pays, etc.
- Un individu peut avoir plusieurs nationalités (toutes les indiquer).
- Les statuts maritaux doit être 'célibataire', 'marié', 'divorcé', 'veuf', etc ou "" si indéfini.
- Le nom de la ville est optionnel.
- Si la ligne "Etat civil" est présente, l'adresse s'y trouve obligatoirement.
- L'adresse doit contenir le nom de la rue et un numéro.	
- Le genre peut être 'Née' ou 'Né' et est déterminé par la présence d'un e à 'née'.
Tâche :
En vous inspirant de l'exemple donné, reconstruisez, s'il vous plait, l'entête du document en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous lisez l'entête à deux niveau du tableau dans le document en reconnaissant les éléments sous la forme ("first level", "second level") :
```plaintext
[("Atelier", ""),
("Occupation", ""),
("Entrée", ""),
("Sortie", ""),
("Présence: Années", ""),
("Présence: Mois", ""),
("Présence: Jours", ""),
("Observations", ""),
("Divers", "")]
```
Contexte :
- Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
- La colonne 'Atelier' peut être absente.
- Pour chaque ligne du tableau, créer une entrée dans le dictionnaire.
- Il est possible que certaines lignes du tableau ne contiennent qu'une observation. Dans ce cas, remplir uniquement la colonne 'Observations' d'une nouvelle entrée.
- Des lignes peuvent avoir des informations manquantes.
- Tu dois ecrire <bis> à la place de '"'.
- "Divers" contient des informations supplémentaires sur le montant de la gratifiaction par exemple (Francs et centimes).
Tâche :
En vous inspirant de l'exemple donné, reconstruisez, s'il vous plait, le tableau en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. Ne confond pas '"' et '11'. Tu dois egalement faire attention à ne pas confondre une ligne est une annotation marginale.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous prenez connaissance des informations du document en essayant de reperer les informations marginales sous la forme ("first level", "second level") :
```plaintext
[("Annotation", "")]
```
Contexte :
- Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
- Les annotations sont des informations supplémentaires sur le document.
- Les annotations peuvent être des notes, des remarques, des précisions, etc.
- Les annotations sont essentiellement manuscrites.
- Les annotations peuvent être sur plusieurs lignes.
Tâche :
En vous inspirant de l'exemple donné, reconstruisez, s'il vous plait, la liste des annotations en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. Veillez également à ne pas mélanger les annotations.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes un agent spécialisé dans l'adaptation des informations précédemment extraites.
En utilisant l'exemple donné, votre mission est de transformer les informations extraites en XML. Pour cela vous avez a votre disposition les tags suivants :
```xml
<Document>
<Nom>
<Genre>
<Statut>
<DateDeNaissance>
<LieuDeNaissance>
<Nationalité>
<Adresse>
<Ville>
<Table>
<Ligne>
<Atelier>
<Occupation>
<Entrée>
<Sortie>
<Présence>
<Années>
<Mois>
<Jours>
<Observations>
<Divers>
<Annotation>
```
Si une balise est vide alors ne pas l'inclure.
"""
            },
        ],
    }
]

In [6]:
# Few shot example

# 2-shot
shot1_im_name = "IMG_1198"
shot2_im_name = "IMG_0690"

im_name = "../data/inputs/A-B/IMG_0023.jpg"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": f"../data/inputs/A-B/{shot1_im_name}.jpg"},
            {
                "type": "text",
                "text": """Voici un exemple de la tâche que vous devez accomplir. Vous devez extraire les informations du document manuscrit et les structurer en XML."""
            }
        ]
    },
    {
      "role": "assistant",
      "content": [
          {
                "type": "text", "text": """Voici la transcription correspondant à l'exemple ci-dessus (XML) :
                    <?xml version="1.0" encoding="utf-8"?>
<Document>
  <Nom>Bouteiller Hélène</Nom>
  <Genre>Née</Genre>
  <NomDuConjoint>
    <added-above>
      <sprited>Vve</sprited>
    </added-above>
     
    <sprited>fme Rayot</sprited>
     Fme Braun
  </NomDuConjoint>
  <DateDeNaissance>6 Octobre 1899</DateDeNaissance>
  <LieuDeNaissance>Hérimoncourt</LieuDeNaissance>
  <Nationalité>française</Nationalité>
  <Ville>
    <pence>Belfort</pence>
  </Ville>
  <Adresse>
    <pence>du Berger, 19</pence>
  </Adresse>
  <Table>
    <Ligne>
      <Occupation>
        Fge 
        <pence>cartonnage</pence>
      </Occupation>
      <Entrée>31 Mars 1924</Entrée>
      <Sortie>5 Juillet 1924</Sortie>
      <Présence>3</Présence>
      <Observations>
        5j 
        <printed>
          <coloured-ink>Gratification 1925</coloured-ink>
        </printed>
      </Observations>
      <Divers>89.-</Divers>
    </Ligne>
    <Ligne>
      <Occupation>- d° -</Occupation>
      <Entrée>21 Juillet 1924</Entrée>
      <Sortie>10 Janvier 1930</Sortie>
      <Présence>55</Présence>
      <Observations>
        21 jours 
        <bis/>
        180e ann.
      </Observations>
      <Divers>100.-</Divers>
    </Ligne>
    <Ligne>
      <Observations>
        <bis/>
        1926
      </Observations>
      <Divers>117.-</Divers>
    </Ligne>
    <Ligne>
      <Observations>
        <bis/>
        1927
      </Observations>
      <Divers>126-</Divers>
    </Ligne>
    <Ligne>
      <Observations>
        <bis/>
        10° an. Arm
      </Observations>
      <Divers>140-</Divers>
    </Ligne>
    <Ligne>
      <Observations>
        <bis/>
        1928
      </Observations>
      <Divers>159-</Divers>
    </Ligne>
    <Ligne>
      <Observations>
        <bis/>
        1929
      </Observations>
      <Divers>182-</Divers>
    </Ligne>
    <Ligne>
      <Observations>quitte Belfort</Observations>
    </Ligne>
  </Table>
  <Annotation>
    <pence>Réfectoire 24</pence>
  </Annotation>
</Document>
"""
            },
      ]
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": f"../data/inputs/A-B/{shot2_im_name}.jpg"},
            {
                "type": "text",
                "text": """Voici un second exemple de la tâche que vous devez accomplir. Vous devez extraire les informations du document manuscrit et les structurer en XML."""
            }
        ]
    },
    {
      "role": "assistant",
      "content": [
          {
                "type": "text", "text": """Voici la transcription correspondant au second exemple ci-dessus (XML) :
                    <?xml version="1.0" encoding="utf-8"?>
                    <Document>
                    <Nom>Bernhard Jeanne Berthe</Nom>
                    <Genre>Née</Genre>
                    <NomDuConjoint>Fme Blind Georg</NomDuConjoint>
                    <DateDeNaissance>19 Mars 1907</DateDeNaissance>
                    <LieuDeNaissance>Etueffont-Bas</LieuDeNaissance>
                    <Nationalité>française</Nationalité>
                    <Ville>
                        <pence>Belfort</pence>
                    </Ville>
                    <Adresse>
                        <pence>de la Marseillaise N°21</pence>
                    </Adresse>
                    <Table>
                        <Ligne>
                        <Occupation>Fge pliage</Occupation>
                        <Entrée>30 Janvier 1922</Entrée>
                        <Sortie>
                            24 Nov
                            <added-above>bre</added-above>
                            1923
                        </Sortie>
                        <Présence>1 9</Présence>
                        <Observations>
                            <added-above>
                            28 jours 
                            <pence>annonce par nous</pence>
                            </added-above>
                            
                            <printed>
                            <coloured-ink>Gratification 1925</coloured-ink>
                            </printed>
                        </Observations>
                        <Divers>61. -</Divers>
                        </Ligne>
                        <Ligne>
                        <Occupation>- M -</Occupation>
                        <Entrée>11 Août 1924</Entrée>
                        <Sortie>7 Janvier 1928</Sortie>
                        <Présence>3 4</Présence>
                        <Observations>
                            <added-above>Pour garder son enfant</added-above>
                            
                            <coloured-ink>Alsthom</coloured-ink>
                            
                            <bis/>
                            180e ann
                        </Observations>
                        <Divers>70. -</Divers>
                        </Ligne>
                        <Ligne>
                        <Occupation>Fin. Pliage</Occupation>
                        <Entrée>1er Avril 1930</Entrée>
                        <Sortie>14 Juin 1930</Sortie>
                        <Présence>- 2</Présence>
                        <Observations>
                            <added-above>le compte a été demandé par le Cx</added-above>
                            14 j. 
                            <bis/>
                            1920
                        </Observations>
                        <Divers>77. -</Divers>
                        </Ligne>
                        <Ligne>
                        <Observations>
                            <bis/>
                            1927
                        </Observations>
                        <Divers>87 -</Divers>
                        </Ligne>
                    </Table>
                    <Annotation>
                        <pence>Touche l'allocation familiale</pence>
                        
                    Pupille de la Nation
                    </Annotation>
                    </Document>
"""
            },
      ]
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": im_name},
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous lisez les informations de l'entête du document en reconnaissant les éléments sous la forme ("first level", "second level") :
```plaintext
[("Nom complet", ""),
("Nom complet du Mari", ""),
("Marqueur", ""),
("Date de naissance", ""),
("Lieu de naissance", ""),
("Nationalité", ""),
("Statut marital", ""),
("Genre", ""),
("Adresse", ""),
("Ville", "")]
```
Contexte :
- Le document est une fiche d'entreprise manuscrite en français datant des années 1900.
- Le nom de la personne est composé d'un ou plusieurs prénoms et d'un nom de famille.
- Les noms des personnes et des villes ne sont pas toujours français.
- Les marqueurs peuvent être 'fme', 'fe', 'vve', 've', 'femme', 'divorcée', etc.
- Les marqueurs précèdent le nom du mari si la personne est une femme mariée.
- Le nom du conjoint est composé d'un ou plusieurs marqueurs et du ou des noms. ex : "fme Jean Dupont", "ve Marie Curie".
- La date de naissance peut être sous la forme "jj/mm/aaaa" ou "jj mois aaaa".
- Le lieu de naissance peut être une ville, un village, un pays, etc.
- Un individu peut avoir plusieurs nationalités (toutes les indiquer).
- Les statuts maritaux doit être 'célibataire', 'marié', 'divorcé', 'veuf', etc ou "" si indéfini.
- Le nom de la ville est optionnel.
- Si la ligne "Etat civil" est présente, l'adresse s'y trouve obligatoirement.
- L'adresse doit contenir le nom de la rue et un numéro.	
- Le genre peut être 'Née' ou 'Né' et est déterminé par la présence d'un e à 'née'.
Tâche :
En vous inspirant des exemples donnés, reconstruisez, s'il vous plait, l'entête du document en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous lisez l'entête à deux niveau du tableau dans le document en reconnaissant les éléments sous la forme ("first level", "second level") :
```plaintext
[("Atelier", ""),
("Occupation", ""),
("Entrée", ""),
("Sortie", ""),
("Présence: Années", ""),
("Présence: Mois", ""),
("Présence: Jours", ""),
("Observations", ""),
("Divers", "")]
```
Contexte :
- Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
- La colonne 'Atelier' peut être absente.
- Pour chaque ligne du tableau, créer une entrée dans le dictionnaire.
- Il est possible que certaines lignes du tableau ne contiennent qu'une observation. Dans ce cas, remplir uniquement la colonne 'Observations' d'une nouvelle entrée.
- Des lignes peuvent avoir des informations manquantes.
- Tu dois ecrire <bis> à la place de '"'.
- "Divers" contient des informations supplémentaires sur le montant de la gratifiaction par exemple (Francs et centimes).
Tâche :
En vous inspirant des exemples donnés, reconstruisez, s'il vous plait, le tableau en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. Ne confond pas '"' et '11'. Tu dois egalement faire attention à ne pas confondre une ligne est une annotation marginale.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous prenez connaissance des informations du document en essayant de reperer les informations marginales sous la forme ("first level", "second level") :
```plaintext
[("Annotation", "")]
```
Contexte :
- Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
- Les annotations sont des informations supplémentaires sur le document.
- Les annotations peuvent être des notes, des remarques, des précisions, etc.
- Les annotations sont essentiellement manuscrites.
- Les annotations peuvent être sur plusieurs lignes.
Tâche :
En vous inspirant des exemples donnés, reconstruisez, s'il vous plait, la liste des annotations en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. Veillez également à ne pas mélanger les annotations.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes un agent spécialisé dans l'adaptation des informations précédemment extraites.
En utilisant les exemples donnés, votre mission est de transformer les informations extraites en XML. Pour cela vous avez a votre disposition les tags suivants :
```xml
<Document>
<Nom>
<Genre>
<Statut>
<DateDeNaissance>
<LieuDeNaissance>
<Nationalité>
<Adresse>
<Ville>
<Table>
<Ligne>
<Atelier>
<Occupation>
<Entrée>
<Sortie>
<Présence>
<Années>
<Mois>
<Jours>
<Observations>
<Divers>
<Annotation>
```
Si une balise est vide alors ne pas l'inclure.
"""
            },
        ],
    }
]

In [7]:
# Convert input into the required format
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

# Tokenize and move to GPU
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

In [8]:
# Generate output
generated_ids = model.generate(**inputs, max_new_tokens=2048)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
]

# Decode and print result
# Decode and extract only the assistant's response
output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
assistant_response = output_text[0].split("assistant\n", 1)[-1]  # Extracts only the assistant's part

print("Generated Output:\n", assistant_response)

del generated_ids, generated_ids_trimmed, inputs, output_text, assistant_response
torch.cuda.empty_cache()

Generated Output:
 Voici la transcription structurée en XML pour le document fourni :

```xml
<Document>
    <Nom>Adolphe Jeanne</Nom>
    <Genre>Née</Genre>
    <DateDeNaissance>9 juillet 1898</DateDeNaissance>
    <LieuDeNaissance>Belfort</LieuDeNaissance>
    <Nationalité>française</Nationalité>
    <Adresse>Rue Lavoueuse 9</Adresse>
    <Ville>Belfort</Ville>
    
    <Table>
        <Ligne>
            <Occupation>Suiveuse For</Occupation>
            <Entrée>26 août 1912</Entrée>
            <Sortie>5 avril 1913</Sortie>
            <Présence>
                <Mois>7 1/3</Mois>
            </Présence>
        </Ligne>
        <Ligne>
            <Occupation>Vr denideule</Occupation>
            <Entrée>25 juin 1923</Entrée>
            <Sortie>27 juin 1923</Sortie>
            <Présence>
                <Mois>-</Mois>
            </Présence>
            <Divers>9</Divers>
        </Ligne>
    </Table>
    
    <Annotation></Annotation>
</Document>
```

### Explications :
1. **Ent

## Run LLM over batches

In [4]:
im_name = ["../data/inputs/IMG_0152.jpg", "../data/inputs/IMG_1029.jpg"]

for i in range(2):
    x = im_name[i]
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": x},
                {
                    "type": "text",
                    "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
    Dans un premier temps, vous lisez les informations de l'entête du document en reconnaissant les éléments sous la forme ("first level", "second level") :
    ```plaintext
    [("Nom complet", ""),
    ("Nom complet du Mari", ""),
    ("Marqueur", ""),
    ("Date de naissance", ""),
    ("Lieu de naissance", ""),
    ("Nationalité", ""),
    ("Statut marital", ""),
    ("Genre", ""),
    ("Adresse", ""),
    ("Ville", "")]
    ```
    Contexte :
    - Le document est une fiche d'entreprise manuscrite en français datant des années 1900.
    - Les noms des personnes et des villes ne font pas toujours français.
    - Les marqueurs peuvent être 'fme', 'fe', 'vve', 've', 'femme', 'divorcée', etc.
    - Les marqueurs précèdent le nom du mari si la personne est une femme mariée.
    - La date de naissance peut être sous la forme "jj/mm/aaaa" ou "jj mois aaaa".
    - Le lieu de naissance peut être une ville, un village, un pays, etc.
    - Un individu peut avoir plusieurs nationalités (toutes les indiquer).
    - Les statuts maritaux doit être 'célibataire', 'marié', 'divorcé', 'veuf', etc ou "" si indéfini.
    - Le nom de la ville est optionnel.
    - Si la ligne "Etat civil" est présente, l'adresse s'y trouve obligatoirement.
    - L'adresse doit contenir le nom de la rue et un numéro.	
    - Le genre peut être 'homme' ou 'femme' et est déterminé par la présence d'un e à 'née'.
    Tâche :
    Reconstruisez, s'il vous plait, l'entête du document en remplissant toutes les information dans le dictionnaire.
    Faites attention à bien lire les mots et chiffres correctement.
    """
                },
                {
                    "type": "text",
                    "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
    Dans un premier temps, vous lisez l'entête à deux niveau du tableau dans le document en reconnaissant les éléments sous la forme ("first level", "second level") :
    ```plaintext
    [("Atelier", ""),
    ("Occupation", ""),
    ("Entrée", ""),
    ("Sortie", ""),
    ("Présence: Années", ""),
    ("Présence: Mois", ""),
    ("Présence: Jours", ""),
    ("Observations", ""),
    ("Divers", "")]
    ```
    Contexte :
    - Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
    - La colonne 'Atelier' peut être absente.
    - Pour chaque ligne du tableau, créer une entrée dans le dictionnaire.
    - Des lignes peuvent avoir des informations manquantes.
    - Tu dois ecrire <bis> à la place de '"' uniquement si ce caractère est seul dans une cellule.
    - "Divers" contient des informations supplémentaires sur le montant de la gratifiaction par exemple (Francs et centimes).
    Tâche :
    Reconstruisez, s'il vous plait, le tableau en remplissant toutes les information dans le dictionnaire.
    Faites attention à bien lire les mots et chiffres correctement. Ne confond pas '"' et '11'. Tu dois egalement faire attention à ne pas confondre une ligne est une annotation marginale.
    """
                },
                {
                    "type": "text",
                    "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
    Dans un premier temps, vous prenez connaissance des informations du document en essayant de reperer les informations marginales sous la forme ("first level", "second level") :
    ```plaintext
    [("Annotation", "")]
    ```
    Contexte :
    - Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
    - Les annotations sont des informations supplémentaires sur le document.
    - Les annotations peuvent être des notes, des remarques, des précisions, etc.
    - Les annotations sont essentiellement manuscrites.
    - Les annotations peuvent être sur plusieurs lignes.
    Tâche :
    Reconstruisez, s'il vous plait, la liste des annotations en remplissant toutes les information dans le dictionnaire.
    Faites attention à bien lire les mots et chiffres correctement. Veillez également à ne pas mélanger les annotations.
    """
                },
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    # Tokenize and move to GPU
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    # Generate output
    generated_ids = model.generate(**inputs, max_new_tokens=2048)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    # Decode and print result
    # Decode and extract only the assistant's response
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    assistant_response = output_text[0].split("assistant\n", 1)[-1]  # Extracts only the assistant's part

    print("Generated Output:\n", assistant_response)

    del generated_ids, generated_ids_trimmed, inputs
    torch.cuda.empty_cache()

Generated Output:
 ### Entête du Document

```plaintext
[
    ("Nom complet", "Gibertini Arnold Mario"),
    ("Nom complet du Mari", ""),
    ("Marqueur", ""),
    ("Date de naissance", "6 Mai 1882"),
    ("Lieu de naissance", "Tomigini"),
    ("Nationalité", "Italienne"),
    ("Statut marital", ""),
    ("Genre", "homme"),
    ("Adresse", "Rue Jean Dollfus 81"),
    ("Ville", "")
]
```

### Tableau du Document

```plaintext
[
    ("Atelier", ""),
    ("Occupation", "Commerciale"),
    ("Entrée", "9 Octobre 1922"),
    ("Sortie", "9 Mars 1929"),
    ("Présence: Années", "6"),
    ("Présence: Mois", "5"),
    ("Présence: Jours", ""),
    ("Observations", "ouvrier de second ordre\nétait auxiliaire à l'étang\ndu 21.8.1922 au 7.\nGratification 1923\n1924\n1925\n780° arm.\n1926\n1927\naccident du travail le 3 avril 1927\nPlaie contuse du coude droit avec décollement de la peau\nEraflures aux deux mains-Forte commotion avec edème\nde la cuisse gauche."),
    ("Divers", "Fr. Cent.\n10. 1922\n

In [4]:
import os
import re
import torch
from pathlib import Path

input_dir = Path("../data/inputs")
output_dir = Path("../data/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Filtrer les fichiers image
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
# Trouver toutes les images dans tous les sous-dossiers
image_files = [p for p in input_dir.rglob("*") if p.suffix.lower() in image_extensions]

system_prompts = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": None},
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous lisez les informations de l'entête du document en reconnaissant les éléments sous la forme ("first level", "second level") :
```plaintext
[("Nom", ""),
("NomDuConjoint", ""),
("DateDeNaissance", ""),
("LieuDeNaissance", ""),
("Nationalité", ""),
("Statut", ""),
("Genre", ""),
("Adresse", ""),
("Ville", "")]
```
Contexte :
- Le document est une fiche d'entreprise manuscrite en français datant des années 1900.
- Les noms des personnes et des villes ne font pas toujours français.
- Les marqueurs peuvent être 'fme', 'fe', 'vve', 've', 'femme', 'divorcée', etc.
- Les marqueurs précèdent le nom du mari si la personne est une femme mariée.
- Le nom du conjoint est composé d'un ou plusieurs marqueurs et du ou des noms.
- La date de naissance peut être sous la forme "jj/mm/aaaa" ou "jj mois aaaa".
- Le lieu de naissance peut être une ville, un village, un pays, etc.
- Un individu peut avoir plusieurs nationalités (toutes les indiquer).
- Les statuts maritaux doit être 'célibataire', 'marié', 'divorcé', 'veuf', etc ou "" si indéfini.
- Le nom de la ville est optionnel.
- Si la ligne "Etat civil" est présente, l'adresse s'y trouve obligatoirement.
- L'adresse doit contenir le nom de la rue et un numéro.	
- Le genre peut être 'Née' ou 'Né' et est déterminé par la présence d'un e à 'née'.
Tâche :
Reconstruisez, s'il vous plait, l'entête du document en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. S'il y a des abréviations, ecrivez-les telles quelles.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous lisez l'entête à deux niveau du tableau dans le document en reconnaissant les éléments sous la forme ("first level", "second level") :
```plaintext
[("Atelier", ""),
("Occupation", ""),
("Entrée", ""),
("Sortie", ""),
("Présence: Années", ""),
("Présence: Mois", ""),
("Présence: Jours", ""),
("Observations", ""),
("Divers", "")]
```
Contexte :
- Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
- La colonne 'Atelier' peut être absente.
- Pour chaque ligne du tableau, créer une entrée dans le dictionnaire.
- Il est possible que certaines lignes du tableau ne contiennent qu'une observation. Dans ce cas, remplir uniquement la colonne 'Observations' d'une nouvelle entrée.
- Des lignes peuvent avoir des informations manquantes.
- Tu dois ecrire <bis> à la place de '"'.
- "Divers" contient des informations supplémentaires sur le montant de la gratifiaction par exemple (Francs et centimes).
Tâche :
Reconstruisez, s'il vous plait, le tableau en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. Ne confond pas '"' et '11'. Tu dois egalement faire attention à ne pas confondre une ligne est une annotation marginale.
S'il y a des abréviations, ecrivez-les telles quelles.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes une IA spécialisée dans l'extraction d'informations de documents historiques manuscrits en français.
Dans un premier temps, vous prenez connaissance des informations du document en essayant de reperer les informations marginales sous la forme ("first level", "second level") :
```plaintext
[("Annotation", "")]
```
Contexte :
- Le document est une fiche d'entreprise textile manuscrite en français datant des années 1900.
- Les annotations sont des informations supplémentaires sur le document.
- Les annotations peuvent être des notes, des remarques, des précisions, etc.
- Les annotations sont essentiellement manuscrites.
- Les annotations peuvent être sur plusieurs lignes.
Tâche :
Reconstruisez, s'il vous plait, la liste des annotations en remplissant toutes les information dans le dictionnaire.
Faites attention à bien lire les mots et chiffres correctement. Veillez également à ne pas mélanger les annotations.
S'il y a des abréviations, ecrivez-les telles quelles.
"""
            },
            {
                "type": "text",
                "text": """Vous êtes un agent spécialisé dans l'adaptation des informations précédemment extraites.
Votre mission est de transformer les informations extraites en XML. Pour cela vous avez a votre disposition les tags suivants :
```xml
<Document>
<Nom>
<Genre>
<Statut>
<DateDeNaissance>
<LieuDeNaissance>
<Nationalité>
<Adresse>
<Ville>
<Table>
<Ligne>
<Atelier>
<Occupation>
<Entrée>
<Sortie>
<Présence>
<Années>
<Mois>
<Jours>
<Observations>
<Divers>
<Annotation>
```

Si une balise est vide alors ne pas l'inclure.
"""
            },
        ],
    }
]

def extract_xml(text):
    # Recherche d'un bloc XML entre des balises markdown ```xml ... ```
    match = re.search(r"```xml\s*(.*?)\s*```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

for image_path in image_files:
    print(f"Processing: {image_path.relative_to(input_dir)}")

    # Générer le chemin de sortie correspondant
    relative_path = image_path.relative_to(input_dir)
    output_subdir = output_dir / relative_path.parent
    output_subdir.mkdir(parents=True, exist_ok=True)
    output_path = output_subdir / (image_path.stem + ".xml")

    # Construire le message
    messages = system_prompts.copy()
    messages[0]["content"][0]["image"] = str(image_path)

    # Préparer le texte d'entrée pour le modèle
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    generated_ids = model.generate(**inputs, max_new_tokens=2048)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    assistant_response = output_text[0].split("assistant\n", 1)[-1]

    xml_content = extract_xml(assistant_response)
    if not xml_content:
        print(f"[⚠️] Aucun bloc XML détecté pour : {relative_path}")
        continue

    # Sauvegarde
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(xml_content)
    print(f"✅ Sauvegardé : {output_path.relative_to(output_dir)}")

    # Nettoyage GPU
    del generated_ids, generated_ids_trimmed, inputs
    torch.cuda.empty_cache()


Processing: A-B/IMG_0927.jpg
✅ Sauvegardé : A-B/IMG_0927.xml
Processing: A-B/IMG_0312.jpg
✅ Sauvegardé : A-B/IMG_0312.xml
Processing: A-B/IMG_0420.jpg
✅ Sauvegardé : A-B/IMG_0420.xml
Processing: A-B/IMG_0708.jpg
✅ Sauvegardé : A-B/IMG_0708.xml
Processing: A-B/IMG_0654.jpg
✅ Sauvegardé : A-B/IMG_0654.xml
Processing: A-B/IMG_0492.jpg
✅ Sauvegardé : A-B/IMG_0492.xml
Processing: A-B/IMG_0096.jpg
✅ Sauvegardé : A-B/IMG_0096.xml
Processing: A-B/IMG_0819.jpg
✅ Sauvegardé : A-B/IMG_0819.xml
Processing: A-B/IMG_0168.jpg
✅ Sauvegardé : A-B/IMG_0168.xml
Processing: A-B/IMG_0873.jpg
✅ Sauvegardé : A-B/IMG_0873.xml
Processing: A-B/IMG_0783.jpg
✅ Sauvegardé : A-B/IMG_0783.xml
Processing: A-B/IMG_0981.jpg
✅ Sauvegardé : A-B/IMG_0981.xml
Processing: A-B/IMG_0564.jpg
✅ Sauvegardé : A-B/IMG_0564.xml
Processing: A-B/IMG_0726.jpg
✅ Sauvegardé : A-B/IMG_0726.xml
Processing: A-B/IMG_1071.jpg
✅ Sauvegardé : A-B/IMG_1071.xml
Processing: A-B/IMG_0023.jpg
✅ Sauvegardé : A-B/IMG_0023.xml
Processing: A-B/IMG_0801